In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn

In [3]:
import import_ipynb

In [8]:
from GPTOSSModel import Model,config

/opt/anaconda3/envs/tf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from tokenizer import get_tokenizer
tokenizer=get_tokenizer()

In [5]:
def text_to_token_ids(text,tokenizer):
    ids=tokenizer.encode(text)
    encoded_ids=torch.tensor(ids).unsqueeze(0)
    return encoded_ids

In [6]:
def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

In [13]:
def generate_new_text(model,device,max_new_tokens=100,prompt="Once upon a day",top_k=50,temperature=0.8,context_length=1024):
    model = model.to(device)
    model.eval()
    prompt=text_to_token_ids(prompt,tokenizer).to(device)
    for _ in range(max_new_tokens):
        input_cond=prompt[:,-context_length:]      #(b,num_tokens)
        with torch.no_grad():
          logits,_=model(input_cond)           #(b,num_tokens,vocab_size)
        last_output_vec=logits[:,-1,:]
        if top_k is not None:
          top_logits,_=torch.topk(last_output_vec,top_k)
          masked_logits=torch.where(last_output_vec<top_logits[:,-1].unsqueeze(1),torch.tensor(float('-inf'),device=last_output_vec.device),last_output_vec)
        else:
          masked_logits=last_output_vec
        masked_logits=masked_logits/temperature
        probs=torch.softmax(masked_logits,dim=-1)
        output_token=torch.multinomial(probs,num_samples=1)
        prompt=torch.cat((prompt,output_token),dim=1)     #(b,num_tokens+1)
    result = token_ids_to_text(prompt,tokenizer) 
    return result

In [15]:
model= Model(config(
    num_heads=8,
    num_kv_groups=4,
    num_experts=4,
    experts_per_token=1,
    num_hidden_layers=16,
    embed_dim =1024,
    intermediate_size = 2048    
    ))


In [16]:
device='mps'

In [17]:
model.load_state_dict(
            torch.load(
                'GPTOSS/best-gpt-oss-params-2run.pt',
                map_location=device,
                weights_only=True
            )
        )

<All keys matched successfully>

In [22]:
generate_new_text(model,device,prompt='how are you')

'how are you playing with, Lily?" The fairy said in a tiny voice. "I\'m a fairy. This is my crown. I found it on the ground. It\'s very pretty tipo."\n\nLily was amazed. She wanted to try the fairy\'s crown. She reached out and took it from the fairy. The fairy smiled and said, "Thank you, Lily. You are very kind. What is your name?"\n\nLily said, "My name is Lily. What is yours?"\n\nThe fairy said, "'